# 클립 추출 파이프라인 (Colab 버전)
ResNet50으로 프레임 피처 추출 → 4프레임 클립 생성 → Google Drive 저장

In [ ]:

# 런타임 끊김 방지 (이 셀 실행 후 백그라운드에서 계속 클릭)
import threading
import time

def keep_alive():
    from IPython.display import display, Javascript
    while True:
        display(Javascript('document.querySelector("#top-toolbar > colab-connect-button").shadowRoot.querySelector("#connect").click()'))
        time.sleep(60)

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print("런타임 끊김 방지 시작")


In [ ]:
# ========================================
# ⚠️ 여기만 수정
MY_MOVIES = [
    '6OvtUA4JPsk',
    'AFiVD7uomv8',
    'AJDngK8lc5Y',
    'HNfa_rRlVl4',
    'nNTui4Lgc5w',
    'pXFG4cVq-Ms',
    'sdIwllo85S4',
    'x2PjLCMrfTk',
    'y97D-Ixgya0',
    'XFcW-PzEZDw',
    'gongjo',
    'gongjo2',
]
SAVE_NAME = 'clips_chaeyeon'
DRIVE_PATH = '/content/drive/MyDrive'  # 본인 드라이브 경로로 수정
BATCH_SIZE = 16
TMP_DIR = '/tmp/frames_cache'
# ========================================

In [ ]:
!pip install torch torchvision huggingface_hub -q

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import re
import os
import shutil
from huggingface_hub import hf_hub_download
from google.colab import drive

drive.mount('/content/drive')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {device}')

resnet = models.resnet50(weights='IMAGENET1K_V1')
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])
resnet.eval().to(device)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print('모델 로드 완료')

In [ ]:
def parse_annotations(movie_id, local_dir):
    """로컬 디렉토리에서 어노테이션 파싱 → (start, end, label) 리스트"""
    txt_path = os.path.join(local_dir, 'annotations', 'raw_txt', f'{movie_id}.txt')
    scenes = []
    with open(txt_path, 'r') as f:
        for line in f:
            match = re.match(r'\[(.+),\s*(\d+),\s*(\d+),\s*(\d+),\s*(\w+)\]', line.strip())
            if match:
                start, end, label = int(match.group(3)), int(match.group(4)), match.group(5)
                if label in ['violence', 'neg_easy', 'neg_hard']:
                    scenes.append((start, end, label))
    return scenes


def collect_clip_frame_keys(movie_id, scenes, clip_len=4, stride=2):
    """클립 구성에 필요한 (movie_id, frame_num) 집합 반환"""
    keys = set()
    for (start, end, label) in scenes:
        frames = list(range(start, end + 1))
        for i in range(0, len(frames) - clip_len + 1, stride):
            for f in frames[i:i + clip_len]:
                keys.add((movie_id, f))
    return keys


def batch_extract_features(frame_keys_paths, batch_size=16):
    """
    frame_keys_paths: [((movie_id, frame_num), path), ...]
    returns: {(movie_id, frame_num): np.array(2048,)}
    """
    features = {}
    total = len(frame_keys_paths)

    for batch_start in range(0, total, batch_size):
        batch = frame_keys_paths[batch_start:batch_start + batch_size]
        imgs, keys = [], []
        for (key, path) in batch:
            keys.append(key)
            if path and os.path.exists(path):
                try:
                    img = Image.open(path).convert('RGB')
                    imgs.append(transform(img))
                    continue
                except:
                    pass
            imgs.append(torch.zeros(3, 224, 224))

        tensor = torch.stack(imgs).to(device)
        with torch.no_grad():
            feats = resnet(tensor).squeeze(-1).squeeze(-1).cpu().numpy()

        for i, key in enumerate(keys):
            features[key] = feats[i]

        if (batch_start // batch_size + 1) % 50 == 0:
            print(f'  피처 추출: {min(batch_start + batch_size, total)}/{total}')

    return features


def build_clips(movie_id, scenes, features, clip_len=4, stride=2, neg_ratio=1.5):
    """사전 추출된 features로 클립 생성 및 neg_ratio 적용"""
    vio_clips, vio_labels = [], []
    neg_clips, neg_labels = [], []

    for (start, end, label) in scenes:
        frames = list(range(start, end + 1))
        for i in range(0, len(frames) - clip_len + 1, stride):
            clip_frames = frames[i:i + clip_len]
            clip_feats = [features.get((movie_id, f), np.zeros(2048)) for f in clip_frames]
            if label == 'violence':
                vio_clips.append(clip_feats)
                vio_labels.append('violence')
            else:
                neg_clips.append(clip_feats)
                neg_labels.append('neg_easy')

    max_neg = int(len(vio_clips) * neg_ratio)
    if len(neg_clips) > max_neg:
        idx = np.random.choice(len(neg_clips), max_neg, replace=False)
        neg_clips = [neg_clips[i] for i in idx]
        neg_labels = [neg_labels[i] for i in idx]

    return vio_clips + neg_clips, vio_labels + neg_labels

In [ ]:
from huggingface_hub import snapshot_download

# ===== Phase 1: 필요한 파일만 골라서 한 번에 다운로드 =====
print('===== Phase 1: HuggingFace snapshot 다운로드 =====')
patterns = (
    [f'frames/{m}/*' for m in MY_MOVIES] +
    [f'annotations/raw_txt/{m}.txt' for m in MY_MOVIES]
)
local_dir = snapshot_download(
    repo_id='DEteam4/datasetVer3',
    repo_type='dataset',
    allow_patterns=patterns,
    local_dir=TMP_DIR
)
print(f'다운로드 완료: {local_dir}')

# ===== Phase 2: 어노테이션 파싱 & 프레임 경로 수집 =====
print('\n===== Phase 2: 어노테이션 파싱 & 프레임 경로 수집 =====')
all_scenes = {}
frame_keys_paths = []

for movie in MY_MOVIES:
    scenes = parse_annotations(movie, local_dir)
    all_scenes[movie] = scenes
    keys = collect_clip_frame_keys(movie, scenes)
    for (movie_id, frame_num) in sorted(keys):
        path = os.path.join(local_dir, 'frames', movie_id, f'frame_{frame_num:06d}.jpg')
        frame_keys_paths.append(((movie_id, frame_num), path if os.path.exists(path) else None))
    print(f'  [{movie}] 씬: {len(scenes)}개 | 필요 프레임: {len(keys)}개')

success = sum(1 for _, p in frame_keys_paths if p)
print(f'\n전체 프레임: {len(frame_keys_paths)}개 | 존재: {success}개')

# ===== Phase 3: 배치 피처 추출 (batch_size=BATCH_SIZE) =====
print(f'\n===== Phase 3: 배치 피처 추출 (batch={BATCH_SIZE}) =====')
features = batch_extract_features(frame_keys_paths, batch_size=BATCH_SIZE)
print(f'피처 추출 완료: {len(features)}개')

# ===== Phase 4: 클립 생성 & Drive 저장 =====
print('\n===== Phase 4: 클립 생성 & 저장 =====')
X_clips, y_clips, movie_ids = [], [], []

for movie in MY_MOVIES:
    clips, labels = build_clips(movie, all_scenes[movie], features)
    X_clips.extend(clips)
    y_clips.extend(labels)
    movie_ids.extend([movie] * len(clips))
    print(f'  [{movie}] {len(clips)}개 (violence={labels.count("violence")}, neg={labels.count("neg_easy")})')

np.save(f'{DRIVE_PATH}/{SAVE_NAME}_X.npy', np.array(X_clips, dtype=np.float32))
np.save(f'{DRIVE_PATH}/{SAVE_NAME}_y.npy', np.array(y_clips))
np.save(f'{DRIVE_PATH}/{SAVE_NAME}_movie_ids.npy', np.array(movie_ids))

print(f'\n===== 완료 =====')
print(f'총 클립: {len(X_clips)}개')
print(f'violence: {y_clips.count("violence")}개')
print(f'neg_easy: {y_clips.count("neg_easy")}개')
print(f'저장 위치: {DRIVE_PATH}/{SAVE_NAME}_*.npy')

# 임시 파일 정리 (필요 시 주석 해제)
# shutil.rmtree(TMP_DIR, ignore_errors=True)
# print('임시 파일 정리 완료')

In [ ]:
# 진행 확인용 (중간에 끊겼을 때)
import numpy as np
y = np.load(f'{DRIVE_PATH}/{SAVE_NAME}_y.npy')
print(f'총 클립: {len(y)}개 | violence: {sum(y=="violence")}개 | neg_easy: {sum(y=="neg_easy")}개')

In [ ]:
# HuggingFace 업로드 (선택사항)
import os
os.environ['HF_TOKEN'] = ''  # 토큰 입력 (또는 Colab Secrets 사용)

!huggingface-cli upload DEteam4/datasetVer3 {DRIVE_PATH}/{SAVE_NAME}_X.npy clips/{SAVE_NAME}_X.npy --repo-type dataset
!huggingface-cli upload DEteam4/datasetVer3 {DRIVE_PATH}/{SAVE_NAME}_y.npy clips/{SAVE_NAME}_y.npy --repo-type dataset
!huggingface-cli upload DEteam4/datasetVer3 {DRIVE_PATH}/{SAVE_NAME}_movie_ids.npy clips/{SAVE_NAME}_movie_ids.npy --repo-type dataset